<style>
  .MathJax, .MathJax_Display, mjx-container, .mjx-chtml, .mjx-math {
    direction: ltr !important;
  }
</style>

<div dir="rtl" style="text-align: right; line-height: 1.6;">

# 📚 קורס ננופוטוניקה, אוניברסיטת בר-אילן 2026
### **מרצה: ד"ר בוריס דסיאטוב**

--- 

## פתרון אופנים אלקטרומגנטיים (Modes) במוליכי גל ננופוטוניים

ברוכים הבאים למדריך פתרון אופנים (Mode-Solving) של הקורס **ננופוטוניקה ופוטוניקה אינטגרטיבית**!

באופטיקה אינטגרטיבית מודרנית, התקדמות האור מונחית באמצעות מבנים תת-אורכי-גל המכונים **מוליכי גל (Waveguides)**. ניתוח מוליכי גל אלו דורש פתרון של משוואות מקסוול בחתך הרוחב הדו-ממדי של המוליך כדי למצוא את **האופנים העצמיים (Eigenmodes)** שלהם (פרופילי שדה אלקטרומגנטי סטציונריים המתקדמים לאורך מוליך הגל ללא שינוי בצורתם).

מחברת זו מספקת מדריך מקיף ומעשי למשתמשים מתחילים לחישוב, ניתוח והשוואה של שתי ארכיטקטורות מוליכי גל מרכזיות:
1. **מוליך גל מלבני (Rectangular Waveguide) מסיליקון:** סוס העבודה הקלאסי של הפוטוניקה בסיליקון המודרנית, הכולא אור באמצעות **החזרה גמורה (Total Internal Reflection - TIR)**.
2. **מוליך גל חריץ (Slot Waveguide) מסיליקון:** ארכיטקטורה בעלת ביצועים גבוהים המנצלת את תנאי הגבול האלקטרומגנטיים כדי להשיג **כליאה והגברה קיצונית של השדה החשמלי** בתוך חריץ צר תת-אורכי-גל בעל מקדם שבירה נמוך.

### 💻 פלטפורמת המידול: Tidy3D מבית Flexcompute
אנו משתמשים בפותר האופנים המתקדם **Tidy3D Mode Solver** כדי לחשב את תכונות האופנים. הניסוח של פותר האופנים ב-Tidy3D פותר בעיית ערכים עצמיים מוכללת דו-ממדית הנגזרת ישירות ממשוואות מקסוול, ומספק חישובים מהירים ומדויקים מאוד של מקדם השבירה האפקטיבי המרוכב ($n_{\text{eff}}$), מקדם שבירת החבורה ($n_g$), שטח האופן האופן האפקטיבי ($A_{\text{eff}}$), והפרופילים המרחביים של כל רכיבי השדה החשמלי והמגנטי ($E_x, E_y, E_z, H_x, H_y, H_z$).

</div>

In [ ]:
import numpy as np
import tidy3d as td
import tidy3d.web as web
from matplotlib import pyplot
from tidy3d.plugins import waveguide
from tidy3d.plugins.mode.web import run as run_mode_solver

# Media used in the examples
si = td.material_library["cSi"]["Li1993_293K"]
sio2 = td.material_library["SiO2"]["Horiba"]

<div dir="rtl" style="text-align: right; line-height: 1.6;">

## 🛠️ חלק 1.1: מוליך גל מלבני (Strip Waveguide) חד-אופן (Single-Mode)

### 🔬 סקירה פיזיקלית ותאורטית
מוליך גל מלבני סטנדרטי מסוג סיליקון על מבודד (SOI) מורכב מליבת **סיליקון (Si)** מלבנית ($n \approx 3.48$ באורך גל $\lambda_0 = 1.55\ \mu\text{m}$) המיוצרת על גבי מצע עבה של **צורן דו-חמצני ($\text{SiO}_2$ - סיליקה)** ($n \approx 1.44$), ומכוסה בחיפוי עליון של $\text{SiO}_2$. ניגודיות מקדמי השבירה הגבוהה הזו ($n_{\text{core}} \approx 3.48$ לעומת $n_{\text{clad}} \approx 1.44$) מאפשרת כליאה אופטית חזקה ורדיוסי כיפוף קטנים במיוחד, המאפשרים אינטגרציה פוטונית צפופה.

### 📐 ממדים גיאומטריים לעבודה במשטר חד-אופן
עבור עבודה במצב חד-אופן (Single-Mode) סטנדרטי, נהוג להשתמש בליבה צרה ברוחב של **$500\text{ nm}$ ($0.5\ \mu\text{m}$)** ועובי של **$220\text{ nm}$ ($0.22\ \mu\text{m}$)**. בממדים אלו, מוליך הגל תומך רק באופנים המרחביים היסודיים עבור הקיטובים האופקיים (quasi-TE) והאנכיים (quasi-TM).

אנו נפתור עבור **שני האופנים** הראשונים (`num_modes=2` ב-`ModeSpec`) כדי לנתח את:
1. **quasi-TE יסודי ($\text{TE}_0$):** ללא נקודות אפס (צמתים) בכיוון האופקי, שדה חשמלי אופקי דומיננטי ($E_y$).
2. **quasi-TM יסודי ($\text{TM}_0$):** ללא נקודות אפס בכיוון האנכי, שדה חשמלי אנכי דומיננטי ($E_z$).

להלן נגדיר גיאומטריה זו באמצעות תוסף מוליכי הגל (waveguide plugin) רב-העוצמה של Tidy3D.

</div>

In [ ]:
strip_single = waveguide.RectangularDielectric(
    wavelength=1.55,
    core_width=0.5,  # Standard single-mode width (500 nm)
    core_thickness=0.22,
    core_medium=si,
    clad_medium=sio2,
    mode_spec=td.ModeSpec(num_modes=2, group_index_step=True),
)

# Take a look at the waveguide cross-section
_ = strip_single.plot_structures(x=0)

<div dir="rtl" style="text-align: right; line-height: 1.6;">

### 🔍 ויזואליזציה של האופנים במוליך גל חד-אופן
בואו נפתור עבור האופנים האופטיים. הפותר מחשב את השדות האלקטרומגנטיים לאורך חתך הרוחב במישור $y-z$. ציר ההתקדמות הוא לאורך כיוון $x$.

עבור כל אופן, אנו משרטטים את רכיב השדה החשמלי הדומיננטי שלו:
- **אופן quasi-TE יסודי (אופן 0):** אנו משרטטים את הרכיב האופקי $E_y$.
- **אופן quasi-TM יסודי (אופן 1):** אנו משרטטים את הרכיב האנכי $E_z$.

</div>

In [ ]:
print(f"Single-Mode Waveguide Effective indices (n_eff): {strip_single.n_eff.values.flatten()}")
print(f"Single-Mode Waveguide Mode areas (A_eff, µm²): {strip_single.mode_area.values.flatten()}")
print(f"Single-Mode Waveguide Group indices (n_group): {strip_single.n_group.values.flatten()}")

fig, ax = pyplot.subplots(1, 2, figsize=(12, 4.5), tight_layout=True)

# Mode 0: Fundamental TE0 (dominant Ey)
strip_single.plot_field("Ey", mode_index=0, ax=ax[0])
ax[0].set_title(f"Fundamental TE0 (Ey), n_eff = {float(strip_single.n_eff.values.flatten()[0]):.4f}")

# Mode 1: Fundamental TM0 (dominant Ez)
strip_single.plot_field("Ez", mode_index=1, ax=ax[1])
ax[1].set_title(f"Fundamental TM0 (Ez), n_eff = {float(strip_single.n_eff.values.flatten()[1]):.4f}")

<div dir="rtl" style="text-align: right; line-height: 1.6;">

## 🚀 חלק 1.2: בואו נעבור למוליך גל רב-אופני (Multimode)!

### 🧠 הפיזיקה של התקדמות רב-אופנית
מה קורה אם נגדיל את ממדי הליבה? בואו נגדיל את רוחב הליבה של מוליך הגל ל-**$1.0\ \mu\text{m}$ ($1000\text{ nm}$)** תוך שמירה על עובי קבוע של **$220\text{ nm}$**.

מכיוון שליבת מוליך הגל רחבה פי שניים, הכליאה המרחבית לאורך הציר האופקי נחלשת ומאפשרת למבנה לתמוך ב-**התקדמות רב-אופנית (Multimode)**. מוליך הגל תומך כעת באופנים מסדר גבוה יותר (אופנים המציגים נקודות אפס או צמתים בתוך הליבה).

אנו נפתור עבור **4 האופנים** הראשונים (`num_modes=4` ב-`ModeSpec`) כדי ללכוד ולאפיין את:
1. **quasi-TE יסודי ($\text{TE}_0$):** ללא נקודות אפס (צמתים) בכיוון האופקי, שדה $E_y$ דומיננטי.
2. **quasi-TE מסדר ראשון ($\text{TE}_1$):** נקודת אפס אחת בכיוון האופקי, שדה $E_y$ דומיננטי.
3. **quasi-TM יסודי ($\text{TM}_0$):** ללא נקודות אפס בכיוון האנכי, שדה $E_z$ דומיננטי.
4. **quasi-TM מסדר ראשון ($\text{TM}_1$):** נקודת אפס אחת בכיוון האנכי, שדה $E_z$ דומיננטי.

</div>

In [ ]:
strip_multi = waveguide.RectangularDielectric(
    wavelength=1.55,
    core_width=1.0,  # Increased core width to 1.0 micron (1000 nm)
    core_thickness=0.22,
    core_medium=si,
    clad_medium=sio2,
    # Solve for 4 modes to capture higher-order spatial profiles
    mode_spec=td.ModeSpec(num_modes=4, group_index_step=True),
)

# Take a look at the waveguide cross-section
_ = strip_multi.plot_structures(x=0)

<div dir="rtl" style="text-align: right; line-height: 1.6;">

### 🔍 ויזואליזציה של האופנים במוליך גל רב-אופני
אנו משרטטים את רכיב השדה החשמלי הדומיננטי עבור כל אחד מ-4 האופנים. שימו לב לצמתים הברורים (אזורי אפס) בכיוון האופקי והאנכי באופנים מסדר ראשון!

</div>

In [ ]:
print(f"Multimode Waveguide Effective indices (n_eff): {strip_multi.n_eff.values.flatten()}")
print(f"Multimode Waveguide Mode areas (A_eff, µm²): {strip_multi.mode_area.values.flatten()}")
print(f"Multimode Waveguide Group indices (n_group): {strip_multi.n_group.values.flatten()}")

fig, ax = pyplot.subplots(2, 2, figsize=(12, 8), tight_layout=True)

# Mode 0: TE0 (dominant Ey)
strip_multi.plot_field("Ey", mode_index=0, ax=ax[0, 0])
ax[0, 0].set_title(f"Mode 0: TE0 (Ey), n_eff = {float(strip_multi.n_eff.values.flatten()[0]):.4f}")

# Mode 1: TE1 (dominant Ey)
strip_multi.plot_field("Ey", mode_index=1, ax=ax[0, 1])
ax[0, 1].set_title(f"Mode 1: TE1 (Ey), n_eff = {float(strip_multi.n_eff.values.flatten()[1]):.4f}")

# Mode 2: TM0 (dominant Ez)
strip_multi.plot_field("Ez", mode_index=2, ax=ax[1, 0])
ax[1, 0].set_title(f"Mode 2: TM0 (Ez), n_eff = {float(strip_multi.n_eff.values.flatten()[2]):.4f}")

# Mode 3: TM1 (dominant Ez)
strip_multi.plot_field("Ez", mode_index=3, ax=ax[1, 1])
ax[1, 1].set_title(f"Mode 3: TM1 (Ez), n_eff = {float(strip_multi.n_eff.values.flatten()[3]):.4f}")

<div dir="rtl" style="text-align: right; line-height: 1.6;">

### 🌟 ויזואליזציה של עוצמת השדה החשמלי הכוללת ($|\mathbf{E}|^2$)
כדי ללכוד את ריכוז האנרגיה האבסולוטי בתוך הליבה, נשרטט את **עוצמת השדה החשמלי הכוללת ($|\mathbf{E}|^2$)**, או $E^2$, המייצגת את התפלגות צפיפות האנרגיה האלקטרומגנטית במרחב:
<span dir="ltr" style="display: block; text-align: center;">$$U_e = \frac{1}{2} \epsilon_0 \epsilon_r |\mathbf{E}|^2$$</span>

שרטוט של $|\mathbf{E}|^2$ מציג את הפיזור המרחבי המוחלט של ההספק האופטי לאורך הליבה והחיפוי, ללא קשר לקיטוב האופן. אנו משיגים זאת על ידי קריאה לפונקציית `plot_field` עם `field_name="E"` ו-`val="abs^2"`.

</div>

In [ ]:
fig, ax = pyplot.subplots(2, 2, figsize=(12, 8), tight_layout=True)

# Mode 0: TE0 |E|²
strip_multi.plot_field("E", val="abs^2", mode_index=0, ax=ax[0, 0])
ax[0, 0].set_title(f"Mode 0: TE0 |E|² Intensity")

# Mode 1: TE1 |E|²
strip_multi.plot_field("E", val="abs^2", mode_index=1, ax=ax[0, 1])
ax[0, 1].set_title(f"Mode 1: TE1 |E|² Intensity")

# Mode 2: TM0 |E|²
strip_multi.plot_field("E", val="abs^2", mode_index=2, ax=ax[1, 0])
ax[1, 0].set_title(f"Mode 2: TM0 |E|² Intensity")

# Mode 3: TM1 |E|²
strip_multi.plot_field("E", val="abs^2", mode_index=3, ax=ax[1, 1])
ax[1, 1].set_title(f"Mode 3: TM1 |E|² Intensity")

<div dir="rtl" style="text-align: right; line-height: 1.6;">

### 📊 ניתוח פרמטרים פיזיקליים: מוליך גל חד-אופן לעומת רב-אופן

בהשוואת התוצאות של מוליך הגל החד-אופן (רוחב $0.5\ \mu\text{m}$) והרב-אופן (רוחב $1.0\ \mu\text{m}$), שימו לב למגמות הפיזיקליות הבאות:

1. **מקדם שבירה אפקטיבי ($n_{\text{eff}}$):**
   - במוליך הגל החד-אופן, לאופן ה-TE0 היסודי יש $n_{\text{eff}} \approx 2.48$, בעוד של-TM0 יש $n_{\text{eff}} \approx 1.83$.
   - במוליך הגל הרב-אופן, מקדם השבירה האפקטיבי של אופן ה-TE0 היסודי מזנק ל-$n_{\text{eff}} \approx 2.85$! תופעה זו מתרחשת מכיוון שליבת הסיליקון הרחבה יותר מכילה הרבה יותר מהאנרגיה החשמלית של האופן בתוכה, מה שמקרב את $n_{\text{eff}}$ למקדם השבירה של סיליקון כחומר נפח ($n_{\text{core}} \approx 3.48$).
   - האופנים המרחביים מסדר גבוה מראים ערכי $n_{\text{eff}}$ נמוכים בהדרגה מכיוון שהשדות המרחביים הרחבים שלהם חודרים עמוק יותר לתוך חיפוי הסיליקה.

2. **מקדם שבירת חבורה ($n_g$):**
   - נפיצות גיאומטרית מבנית חזקה גורמת למקדם שבירת החבורה $n_g$ (הקובע את מהירות התפשטות האותות: $v_g = c_0/n_g$) להיות גדול משמעותית ממקדם השבירה החומרי של סיליקון עצמו.

3. **שטח אופן אפקטיבי ($A_{\text{eff}}$):**
   - מעבר למוליך גל רחב יותר מאפשר לאופן היסודי להצטמצם בצורה הדוקה יותר בתוך הליבה, מה שמקטין את שטח האופן האפקטיבי שלו, בעוד שלאופנים מסדר גבוה יש שטחים אפקטיביים גדולים בהרבה בשל התפרסותם המרחבית ונוכחות צמתים מבניים.

</div>

In [ ]:
# Simple verification check
print("Multimode waveguide successfully solved for 4 guided modes.")

<div dir="rtl" style="text-align: right; line-height: 1.6;">

## ⚡ חלק 2: מוליך גל חריץ מסיליקון (Silicon Slot Waveguide)

### 🧠 הפיזיקה של כליאה בחריץ (Almeida et al., 2004)
במוליך גל מלבני קונבנציונלי, האור כלוא בתוך ליבת הסיליקון בעלת מקדם השבירה הגבוהה באמצעות החזרה גמורה. עם זאת, אם נמקם שתי ליבות סיליקון (הנקראות "מסילות" - Rails) קרוב מאוד אחת לשנייה (במרחק תת-אורכי-גל, לרוב $50 - 100\text{ nm}$), מתרחשת תופעה פיזיקלית ייחודית מאוד.

על פי האלקטרודינמיקה הקלאסית, הרכיב הניצב של **שדה ההעתק החשמלי ($\mathbf{D}$)** חייב להיות רציף במעבר על פני ממשק דיאלקטרי:
<span dir="ltr" style="display: block; text-align: center;">$$D_{1,n} = D_{2,n} \implies \epsilon_1 E_{1,n} = \epsilon_2 E_{2,n}$$</span>

אם נבטא תנאי גבול זה באמצעות מקדם השבירה ($n = \sqrt{\epsilon}$), נקבל:
<span dir="ltr" style="display: block; text-align: center;">$$n_1^2 E_{1,n} = n_2^2 E_{2,n} \implies E_{2,n} = \left( \frac{n_1}{n_2} \right)^2 E_{1,n}$$</span>

אם אזור 1 הוא **סיליקון** ($n_{\text{core}} \approx 3.48$) ואזור 2 הוא **silica** ($n_{\text{slot}} \approx 1.44$), אזי בממשקים האנכיים (שבהם הכיוון הניצב הוא כיוון $y$):
<span dir="ltr" style="display: block; text-align: center;">$$E_{\text{slot}} = \left( \frac{3.48}{1.44} \right)^2 E_{\text{Si}} \approx 5.83 \times E_{\text{Si}}$$</span>

המשמעות היא שהשדה החשמלי האופקי $E_y$ חווה **קפיצה עצומה של כמעט פי 6** באמפליטודה בתוך אזור החריץ בעל מקדם השבירה הנמוך! מכיוון שרוחב החריץ הוא צר במיוחד, השדות משני הממשקים מתאבכים באופן בונה, מה שמביא להגברה חזקה במיוחד של העוצמה האופטית ולכליאה מרחבית הדוקה בתוך החריץ בעל **מקדם השבירה הנמוך**.

תופעה זו הופכת את מוליכי גל החריץ למועמדים מצוינים עבור:
- **חישה אופטית וחישה ביולוגית (Sensing):** אינטראקציה ישירה של השדה עם מולקולות המוחדרות לחריץ.
- **מודולטורים אלקטרו-אופטיים:** הספגת החריץ בפולימרים אלקטרו-אופטיים או חומרים דו-ממדיים.
- **אופטיקה לא-ליניארית:** ניצול עוצמת השדה הגבוהה להמרת תדרים יעילה.

בואו נגדיר את מוליך גל החריץ ב-Tidy3D! נבנה שתי מסילות סיליקון מקבילות המופרדות על ידי חריץ של $100\text{ nm}$, מוקפות בחיפוי רקע של $	ext{SiO}_2$.

</div>

In [ ]:
# Geometry parameters
w_rail = 0.22      
h_rail = 0.22      
w_slot = 0.1       
wavelength = 1.55

# Calculate positions
offset = (w_rail + w_slot) / 2

rail_left = td.Structure(
    geometry=td.Box(center=(0, -offset, 0), size=(td.inf, w_rail, h_rail)),
    medium=si
)

rail_right = td.Structure(
    geometry=td.Box(center=(0, offset, 0), size=(td.inf, w_rail, h_rail)),
    medium=si
)

# Simulation domain
sim_size = (0.1, 3.0, 2.0) 
mode_plane = td.Box(center=(0, 0, 0), size=(0, 3.0, 2.0))

# Fixed Simulation call: added wavelength to grid_spec
sim = td.Simulation(
    size=sim_size,
    grid_spec=td.GridSpec.auto(min_steps_per_wvl=40, wavelength=wavelength),
    structures=[rail_left, rail_right],
    medium=sio2, 
    run_time=1e-12,
)

mode_solver = td.plugins.mode.ModeSolver(
    simulation=sim,
    plane=mode_plane,
    mode_spec=td.ModeSpec(num_modes=2, target_neff=2.0),
    freqs=[td.C_0 / wavelength],
)

<div dir="rtl" style="text-align: right; line-height: 1.6;">

### 🗺️ ויזואליזציה של חתך מוליך גל החריץ
לפני שנריץ את הפותר, בואו נוודא שהמבנה הגיאומטרי שלנו נכון. נשרטט את חתך מקדם השבירה/הפרמיטיביות של המבנה כדי לוודא ששתי מסילות הסיליקון ממוקמות בצורה נכונה ומופרדות על ידי החריץ ברוחב $100\text{ nm}$.

</div>

In [ ]:
# Take a look at the waveguide cross-section
_ = sim.plot_structures(x=0)

<div dir="rtl" style="text-align: right; line-height: 1.6;">

### 🏃‍♂️ הרצת פותר אופני התנודה
כעת נריץ את פותר הערכים העצמיים כדי לחשב את אופני התנודה של מוליך גל החריץ. נשרטט את רכיב השדה החשמלי הרוחבי האופקי ($E_y$).

הביטו מקרוב על המעברים בממשקים הדיאלקטריים! תוכלו להבחין בביטוי פיזיקלי ברור מאוד של תנאי הגבול האלקטרומגנטיים: עוצמת השדה החשמלי מזנקת באופן דרמטי בתוך אזור החריץ בעל מקדם השבירה הנמוך.

</div>

In [ ]:
# Run the solver
mode_data = mode_solver.solve()

# Access results
print(f"Effective indices: {mode_data.n_eff.values.flatten()}")

# Plot horizontal field component Ey
mode_solver.plot_field("Ey", mode_index=0)

<div dir="rtl" style="text-align: right; line-height: 1.6;">

### 🌟 שרטוט גודל השדה החשמלי הכולל ($|\mathbf{E}|$)
כדי להעריך את ריכוז האנרגיה הקיצוני, נשרטט את גודל השדה החשמלי הכולל:
<span dir="ltr" style="display: block; text-align: center;">$$|\mathbf{E}| = \sqrt{|E_x|^2 + |E_y|^2 + |E_z|^2}$$</span>

שימו לב כיצד האנרגיה האופטית ממוקמת כמעט לחלוטין בתוך החריץ המרכזי שרוחבו $100\text{ nm}$. זהו הישג יוצא דופן בננופוטוניקה: הנחיה ודחיסה של אור לתוך אזור הקטן בהרבה מגבול הדיפרקציה של האור עצמו!

</div>

In [ ]:
mode_solver.plot_field("E", mode_index=0)

<div dir="rtl" style="text-align: right; line-height: 1.6;">

## 📝 נקודות מפתח לסיכום ותרגילי סטודנטים

### 🎓 מושגי יסוד שנלמדו
1. **מוליכי גל מלבניים** כולאים אור באמצעות **החזרה גמורה (TIR)**. תכנון חד-אופן סטנדרטי (רוחב $500\text{ nm}$) תומך רק באופנים היסודיים TE0 ו-TM0. הגדלת הרוחב ל-$1.0\ \mu\text{m}$ גורמת למוליך הגל לתמוך באופנים מרחביים מסדר גבוה יותר (TE1, TM1).
2. **עוצמת השדה החשמלי ($E^2$)** מייצגת את הפיזור המרחבי המוחלט של האנרגיה האלקטרומגנטית, הפרופורציונלית לצפיפות האנרגיה $U_e$.
3. **מוליכי גל חריץ** כולאים אור באמצעות **רציפות רכיב שדה ההעתק החשמלי הניצב ($D_n$)** בממשקים דיאלקטריים בעלי ניגודיות גבוהה, מה שגורם לקפיצת שדה עצומה באזור החריץ הצר.

---

### ✏️ שיעורי בית ותרגילי למידה עצמית לסטודנטים

#### 🔬 תרגיל 1: סריקת רוחב מוליך גל מלבני וניתוח תנאי הקיטוע (Cutoff)
- **משימה:** שמרו על עובי ליבה קבוע של $h = 220\text{ nm}$. סרקו את רוחב הליבה $w$ מ-$300\text{ nm}$ עד ל-$1200\text{ nm}$ בצעדים של $100\text{ nm}$.
- **שאלות:**
  1. באיזה רוחב ליבה מדויק מוליך הגל עובר ממשטר של אופן יחיד (single-mode) למשטר רב-אופני (multimode)?
  2. שרטטו את מקדם השבירה האפקטיבי $n_{\text{eff}}$ של 4 האופנים הראשונים כפונקציה של הרוחב. מדוע מקדמי השבירה עולים ככל שהרוחב גדל?

#### 🔬 תרגיל 2: ברירת קיטוב במוליכי גל חריץ
- **משימה:** שרטטו את פרופיל השדה של האופן השני שנפתר (אופן quasi-TM) במוליך גל החריץ.
- **שאלות:**
  1. מדוע אין הגברת שדה בחריץ עבור אופן ה-quasi-TM?
  2. *רמז:* חשבו איזה רכיב של השדה החשמלי ($E_y$ או $E_z$) ניצב לממשקים האנכיים של המסילות, וכתבו את תנאי הגבול המתאים.

</div>